In [26]:
from datasets import load_dataset
import pandas as pd
import re

In [2]:
ds = load_dataset("christinacdl/clickbait_detection_dataset")
ds

README.md:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

train.json:   0%|          | 0.00/2.99M [00:00<?, ?B/s]

val.json:   0%|          | 0.00/375k [00:00<?, ?B/s]

test.json:   0%|          | 0.00/373k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30296 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3787 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3787 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text_label', 'text', 'label'],
        num_rows: 30296
    })
    validation: Dataset({
        features: ['text_label', 'text', 'label'],
        num_rows: 3787
    })
    test: Dataset({
        features: ['text_label', 'text', 'label'],
        num_rows: 3787
    })
})

In [14]:
train = ds["train"].to_pandas()
train.shape
train.head(10)

,text_label,text,label
0,CLICKBAIT,14 Times People Turned Into Emojis At The Munk...,1
1,NOT,BBC to cut Electric Proms for financial reasons,0
2,CLICKBAIT,28 Adorable Animal Things You Need In Your Life,1
3,NOT,Sotomayor on the Issues,0
4,CLICKBAIT,9 Ways To Save Your Sanity On The Wednesday Be...,1
5,NOT,Subpoena to a Lawmaker Is Reported,0
6,NOT,Ask HN: Ideas for mobile marathon?,0
7,NOT,Huge fire in Chilean jail kills 81; 21 injured,0
8,NOT,Dominique Strauss-Kahn refused bail after appe...,0
9,NOT,Open software developers meet at FOSDEM 2008,0


In [15]:
print(ds)
train.shape

DatasetDict({
    train: Dataset({
        features: ['text_label', 'text', 'label'],
        num_rows: 30296
    })
    validation: Dataset({
        features: ['text_label', 'text', 'label'],
        num_rows: 3787
    })
    test: Dataset({
        features: ['text_label', 'text', 'label'],
        num_rows: 3787
    })
})


(30296, 3)

In [16]:
train["label"].value_counts()

label
1    16016
0    14280
Name: count, dtype: int64

In [17]:
train["label"].value_counts(normalize=True)

label
1    0.528651
0    0.471349
Name: proportion, dtype: float64

In [18]:
train["text"].duplicated().sum()

np.int64(0)

In [19]:
train.isnull().sum()

text_label    0
text          0
label         0
dtype: int64

In [23]:
pd.set_option("display.max_colwidth", None)
print(train[train["label"] == 1].sample(20, random_state=42)["text"])
print("---")
print(train[train["label"] == 0].sample(20, random_state=42)["text"])

20331                We Know Your Favorite Ed Sheeran Song Based On The Taylor Swift You Choose
14475                        Which Golden Globe Reaction Face Are You Based On Your Zodiac Sign
16500                                           What Lazy Costume Should You Wear For Halloween
28915                                   13 Awkward Moments When You Have Depression And Anxiety
22073                                                      What No One Tells You About Marriage
26213                                                33 first times every woman has experienced
21518                                         21 Photos Of Jake Gyllenhaal That Really Hit Home
9859                          25 Delightfully Cozy Gifts For Anyone Who Hates Leaving The House
23589                             This Fake Black Friday Movie Trailer Needs To Be A Real Thing
6054                       We Need To Talk About The Biggest Problem With Buying Things In 2015
22642                                   

In [25]:
train["n_words"] = train["text"].str.split().str.len()
train.groupby("label")["n_words"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,14280.0,8.141737,2.400614,1.0,7.0,8.0,10.0,21.0
1,16016.0,10.085539,2.859678,1.0,8.0,10.0,12.0,28.0


In [27]:
train["has_you"] = train["text"].str.contains(r"\byou\b|\byour\b", case=False, regex=True)
train.groupby("label")["has_you"].mean()

label
0    0.014846
1    0.378434
Name: has_you, dtype: float64

In [28]:
train["starts_with_number"] = train["text"].str.match(r"^\d+\s")
train.groupby("label")["starts_with_number"].mean()

label
0    0.017927
1    0.306631
Name: starts_with_number, dtype: float64

## EDA Findings

- Dataset: 30,296 headlines (train split), balanced (53% clickbait / 47% not clickbait),
  no duplicates, no missing values.
- Clickbait headlines are systematically longer (median 10 words vs 8 for non-clickbait).
- The pronoun "you"/"your" appears in 37.8% of clickbait headlines vs 1.5% of non-clickbait
  — the strongest of the three signals found.
- A leading number ("13 Things...", "21 Photos...") appears in 30.7% of clickbait headlines
  vs 1.8% of non-clickbait.
- These three linguistic features (length, presence of "you"/"your", leading number) will
  form the basis of a simple FeatureExtractor for the classical baseline models
  (TF-IDF + LogReg and Random Forest).